# Solution - Audit and rescue the rent model

Instructor solution. The starter `buggy_rent_model.ipynb` hides **ten** methodology
bugs (L04-L07). Below: the filled audit table (Part 1), the corrected pipeline
(Part 2), and the honest model + model card (Part 3).

**Key teaching point:** most of these bugs **barely move the headline number** -
a model can be riddled with errors and still print R² ≈ 0.85. You catch them by
reading the code against the rules, not by hunting for a value that looks wrong.
On this data only the **leakage** and the **reporting** bugs distort the result;
the rest are still wrong and bite on other data, fixed hyperparameters, or
interpretation.

## Part 1 - Audit

| # | Bug (where in the starter) | Rule | Why it's wrong (and its real impact here) |
|---|---|---|---|
| **1** | `locality_mean_rent = groupby("Area Locality")["Rent"].transform("mean")` and `ohe.fit_transform` both on the **full df, before the split** | **L04** leakage | the locality mean is built from test rows' rents (and the row's own), so the model peeks at the target. **Inflates the sealed-test log-R² by ~0.03.** |
| **2** | alpha chosen by `r2_score(y_test, ...)` in the sweep | **L04** | model selection peeks at the test set, so it is no longer an honest hold-out. (Here the peeked alpha = the CV choice, so ~0 gain - but invalid in principle.) |
| **3** | `Ridge` fit on **unscaled** features (no `StandardScaler`) | **L05** | the penalty is fair only by luck. No score impact once alpha is tuned (Ridge ≈ OLS here), but it breaks interpretation (next row) and bites the moment regularization actually grips. |
| **4** | alpha grid `[0.1, 0.2, ..., 1.0]` - linear, narrow | **L06** | the winner sits at the **boundary** (1.0); you can't tell the true optimum isn't outside the range. Use log-uniform + widen. |
| **5** | feature importance read off **unscaled** `\|coef\|` | **L05/L07** | `Size` and `locality_mean_rent` get tiny coefs (their values are in the thousands) so they rank **last** - the opposite of the truth. Magnitude ≠ importance unless features share a scale. |
| **6** | RMSE/MAE reported on the **log** target ("0.36"); R² compared across two y-subsets | **L07** | RMSE 0.36 is log units (~84k in rent units). And log-R² is 0.85 full vs ~0.78 normal-flats: R² shifts with the range, so the comparison is meaningless. |
| **7** | `r2_score(y_train, ...)` reported as model quality | **L04** | training error always looks good (it's what we fit). Says nothing about generalization - the chapter's opening lesson. |
| **8** | `cross_val_score(scoring="neg_mean_squared_error")` printed as "MSE" | **L07** | sklearn returns the **negative** MSE (it maximizes by convention), so "MSE = -0.154" is nonsense. Negate it, or use a positive scorer. |
| **9** | the "FINAL" number is a CV score on the **leaked training** matrix; no sealed test | **L04/L06** | there is no untouched evaluation anywhere - the CV reuses leaked, selection-driving data. Report a sealed test once (or nested CV). |
| **10** | **no `random_state`** on the split, KFold, or model | **L04** | every run reshuffles → a different "best alpha" and score each time. Not reproducible, can't debug, can't compare runs. Set the seed everywhere. |

**Bonus catch:** no `DummyRegressor` baseline, so "R² = 0.85" has no reference point.

## Part 2 - The fix

Seed everything; split first; all preprocessing in a `Pipeline` (refit per fold -
no leakage, scaled before the penalty); alpha tuned by CV on **train only** over a
wide log range; test set sealed.

In [1]:
import numpy as np
import pandas as pd
import warnings
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.metrics import (r2_score, mean_squared_error,
                             mean_absolute_error, median_absolute_error)

SEED = 509                       # fixes bug 10 - reproducible everywhere
np.random.seed(SEED)

df = pd.read_csv("../01_regression_intro/data/House_Rent_Dataset.csv")

# leaky locality feature dropped; raw columns only - all transforms go in the pipeline
num = ["BHK", "Size", "Bathroom"]
cat = ["City", "Furnishing Status", "Area Type"]
X = df[num + cat]
y_raw = df["Rent"].values
y_log = np.log1p(y_raw)

X_tr, X_te, ylog_tr, ylog_te, yraw_tr, yraw_te = train_test_split(
    X, y_log, y_raw, test_size=0.2, random_state=SEED)        # seeded split (fixes 1, 10)
print("train:", X_tr.shape, " test:", X_te.shape, " (test sealed until Part 3)")

train: (3796, 6)  test: (950, 6)  (test sealed until Part 3)


In [2]:
pre = ColumnTransformer([
    ("num", StandardScaler(), num),                          # scale before the penalty (fixes 3)
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), cat),
])
pipe = Pipeline([("pre", pre), ("ridge", Ridge())])

grid = {"ridge__alpha": np.logspace(-3, 3, 13)}             # log-uniform, wide (fixes 4)
gs = GridSearchCV(pipe, grid, cv=KFold(5, shuffle=True, random_state=SEED),
                  scoring="r2")                              # CV on train, not the test set (fixes 2)
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="Found unknown categories")
    gs.fit(X_tr, ylog_tr)

best = gs.best_params_["ridge__alpha"]
print(f"best alpha = {best:g}   CV R2(log) = {gs.best_score_:.3f}   "
      f"interior to range: {1e-3 < best < 1e3}")

best alpha = 1   CV R2(log) = 0.790   interior to range: True


### Feature importance, done right (fixes bug 5)

The starter ranked features by unscaled `|coef|`. With features on a common scale,
`Size` jumps from *last* to a top driver - the ranking inverts.

In [3]:
# same features as the starter, fit unscaled vs scaled, compare |coef| ranking
from sklearn.linear_model import Ridge as _R
feats = num + ["locality_mean_rent"]
Xx = df[num].copy()
Xx["locality_mean_rent"] = df.groupby("Area Locality")["Rent"].transform("mean")
yy = y_log
unscaled = _R(alpha=1.0).fit(Xx.values, yy).coef_
scaled = _R(alpha=1.0).fit(StandardScaler().fit_transform(Xx.values), yy).coef_
cmp = pd.DataFrame({"|coef| unscaled": np.abs(unscaled),
                    "|coef| scaled": np.abs(scaled)}, index=feats)
print(cmp.round(4))
print("\nUnscaled rank (most->least):", list(cmp["|coef| unscaled"].sort_values(ascending=False).index))
print("Scaled rank   (most->least):", list(cmp["|coef| scaled"].sort_values(ascending=False).index))

                    |coef| unscaled  |coef| scaled
BHK                          0.1193         0.0992
Size                         0.0001         0.0540
Bathroom                     0.3669         0.3247
locality_mean_rent           0.0000         0.4509

Unscaled rank (most->least): ['Bathroom', 'BHK', 'Size', 'locality_mean_rent']
Scaled rank   (most->least): ['locality_mean_rent', 'Bathroom', 'BHK', 'Size']


## Part 3 - Honest model + model card

A `DummyRegressor` floor, then the **single** sealed-test evaluation (not the
train score), reported in rent units with positive numbers.

In [4]:
dummy_cv = cross_val_score(DummyRegressor(strategy="mean"), X_tr, ylog_tr,
                           cv=KFold(5, shuffle=True, random_state=SEED),
                           scoring="r2").mean()
print(f"DummyRegressor CV R2(log) = {dummy_cv:.3f}   (our model must clear this)")

DummyRegressor CV R2(log) = -0.001   (our model must clear this)


In [5]:
# THE one-time test evaluation (fixes 7, 9) - in rent units, positive (fixes 6, 8)
p_log = gs.predict(X_te)
p_raw = np.expm1(p_log)
logR2 = r2_score(ylog_te, p_log)
mae   = mean_absolute_error(yraw_te, p_raw)
medae = median_absolute_error(yraw_te, p_raw)
rmse  = mean_squared_error(yraw_te, p_raw) ** 0.5
print(f"log-R2 = {logR2:.3f}")
print(f"MAE    = {mae:,.0f} rent units")
print(f"MedAE  = {medae:,.0f} rent units   <- robust 'typical error'")
print(f"RMSE   = {rmse:,.0f} rent units   <- inflated by the luxury tail")

log-R2 = 0.799
MAE    = 11,364 rent units
MedAE  = 3,639 rent units   <- robust 'typical error'
RMSE   = 31,807 rent units   <- inflated by the luxury tail


In [6]:
# bug 6b demonstrated: R2 depends on the y-range -> comparing across ranges is meaningless
lux = yraw_te >= 100_000
print(f"raw-R2 full test            = {r2_score(yraw_te, p_raw):.2f}")
print(f"raw-R2 normal flats (<100k) = {r2_score(yraw_te[~lux], p_raw[~lux]):.2f}")
print(f"MedAE normal (<100k) = {median_absolute_error(yraw_te[~lux], p_raw[~lux]):,.0f}   "
      f"luxury (>=100k) = {median_absolute_error(yraw_te[lux], p_raw[lux]):,.0f}  ({lux.sum()} flats)")

raw-R2 full test            = 0.58
raw-R2 normal flats (<100k) = 0.63
MedAE normal (<100k) = 3,352   luxury (>=100k) = 59,029  (57 flats)


In [7]:
print("=" * 54)
print(" MODEL CARD - Rent predictor (honest)")
print("=" * 54)
print(f" Data        : House_Rent_Dataset.csv (n={len(df)}), target log1p(Rent)")
print(f" Model       : Ridge(alpha={best:g}) on scaled numerics + OHE categoricals, seed {SEED}")
print(f" Selection   : 5-fold CV on train; test evaluated ONCE")
print(f" Baseline    : DummyRegressor R2(log) = {dummy_cv:.3f}")
print(f" Test scores : log-R2 = {logR2:.3f}   MAE = {mae:,.0f}   MedAE = {medae:,.0f}   RMSE = {rmse:,.0f}")
print(f" Headline    : typical error (MedAE) ~ {medae:,.0f} rent units")
print(" Failure mode: under-predicts luxury flats (>=100k); RMSE tail-dominated")
print(" Report MedAE/MAE in rent units, not log-RMSE or raw-R2, for this skewed target")
print("=" * 54)


 MODEL CARD - Rent predictor (honest)
 Data        : House_Rent_Dataset.csv (n=4746), target log1p(Rent)
 Model       : Ridge(alpha=1) on scaled numerics + OHE categoricals, seed 509
 Selection   : 5-fold CV on train; test evaluated ONCE
 Baseline    : DummyRegressor R2(log) = -0.001
 Test scores : log-R2 = 0.799   MAE = 11,364   MedAE = 3,639   RMSE = 31,807
 Headline    : typical error (MedAE) ~ 3,639 rent units
 Failure mode: under-predicts luxury flats (>=100k); RMSE tail-dominated
 Report MedAE/MAE in rent units, not log-RMSE or raw-R2, for this skewed target


## Conclusion

After fixing all ten bugs, the honest sealed-test **log-R² is ~0.80** - a touch
below the teammate's contaminated number, far above the dummy baseline (~0).
The model is good for typical flats and **under-predicts luxury ones**.

The thing to internalise: **most of the ten bugs never moved the headline number**
(the leak and the log-units reporting did). The rest - no seed, unscaled fit,
unscaled importance, boundary grid, train-R², neg-MSE, CV-as-final - printed a
respectable 0.85 while being wrong. That is exactly why you audit the *code*
against the rules, not the score.

**Bonus directions:** nested CV; target-encode `Area Locality` correctly inside CV
folds; permutation importance; a P10-P90 prediction interval; per-city errors.